# Phase 4 — Silver Validation & Cleansing (Job Notebook)

This notebook validates the Bronze source tables, creates accepted Silver tables, preserves rejected and quarantined records for audit, flags delivery exceptions, and verifies saved results.

**Initial job mode:** `input_batch=all` performs a full refresh from all Bronze rows. A specific batch is deliberately rejected for now: batch-only writes require incremental merge logic and referential-scope rules to avoid overwriting previously processed Silver data or misclassifying cross-batch references.

The Bronze tables are read-only. Silver outputs are overwritten only after all transformation DataFrames have been prepared.


In [0]:
# 1. Job parameters and shared configuration

from pyspark.sql import functions as F
from pyspark.sql.window import Window

dbutils.widgets.text("catalog", "dbwteam3")
dbutils.widgets.text("schema", "order_fulfillment")
dbutils.widgets.text("input_batch", "all")

CATALOG = dbutils.widgets.get("catalog").strip()
SCHEMA = dbutils.widgets.get("schema").strip()
input_batch = dbutils.widgets.get("input_batch").strip()
TARGET = f"{CATALOG}.{SCHEMA}"

if not CATALOG or not SCHEMA or not input_batch:
    raise ValueError("catalog, schema, and input_batch must not be empty.")

if input_batch.lower() != "all":
    raise ValueError(
        "This version supports input_batch='all' only. "
        "Batch-specific processing needs incremental MERGE logic before it is safe."
    )

print("Silver validation job started.")
print("Catalog:", CATALOG)
print("Schema:", SCHEMA)
print("Input batch:", input_batch)
print("Target:", TARGET)


Silver validation job started.
Catalog: dbwteam3
Schema: order_fulfillment
Input batch: all
Target: dbwteam3.order_fulfillment


In [0]:
# 2. Load Bronze sources (read-only)

bronze_table_names = [
    "bronze_pharmacies",
    "bronze_orders",
    "bronze_order_lines",
    "bronze_shipment_lines",
    "bronze_delivery_events"
]

missing_tables = [
    name for name in bronze_table_names
    if not spark.catalog.tableExists(f"{TARGET}.{name}")
]
if missing_tables:
    raise ValueError(f"Required Bronze tables are missing: {missing_tables}")

pharmacies_bronze = spark.table(f"{TARGET}.bronze_pharmacies")
orders_bronze = spark.table(f"{TARGET}.bronze_orders")
order_lines_bronze = spark.table(f"{TARGET}.bronze_order_lines")
shipment_lines_bronze = spark.table(f"{TARGET}.bronze_shipment_lines")
delivery_events_bronze = spark.table(f"{TARGET}.bronze_delivery_events")

print("=== BRONZE INPUT COUNTS ===")
for name, df in [
    ("bronze_pharmacies", pharmacies_bronze),
    ("bronze_orders", orders_bronze),
    ("bronze_order_lines", order_lines_bronze),
    ("bronze_shipment_lines", shipment_lines_bronze),
    ("bronze_delivery_events", delivery_events_bronze),
]:
    print(f"{name}: {df.count()}")


=== BRONZE INPUT COUNTS ===
bronze_pharmacies: 100
bronze_orders: 2080
bronze_order_lines: 4865
bronze_shipment_lines: 5958
bronze_delivery_events: 6198


In [0]:

# PHASE 4 - STEP 5A: Validate and standardize orders
# This cell creates DataFrames only; it does not write tables.


# Known pharmacy IDs
valid_pharmacy_ids = (
    pharmacies_bronze
    .select(F.trim(F.col("pharmacy_id")).alias("pharmacy_id"))
    .filter(
        F.col("pharmacy_id").isNotNull()
        & (F.col("pharmacy_id") != "")
    )
    .distinct()
)

# Standardize order fields and retain original metadata
orders_prepared = (
    orders_bronze
    .withColumn("_order_id_clean", F.trim(F.col("order_id")))
    .withColumn("_pharmacy_id_clean", F.trim(F.col("pharmacy_id")))
    .withColumn(
        "order_time",
        F.expr("try_cast(order_time AS TIMESTAMP)")
    )
    .withColumn(
        "promised_delivery_time",
        F.expr("try_cast(promised_delivery_time AS TIMESTAMP)")
    )
    .join(
        valid_pharmacy_ids.withColumnRenamed(
            "pharmacy_id", "_known_pharmacy_id"
        ),
        F.col("_pharmacy_id_clean") == F.col("_known_pharmacy_id"),
        "left"
    )
)

# Assign rejection reasons
orders_checked = orders_prepared.withColumn(
    "validation_reason",
    F.when(
        F.col("_order_id_clean").isNull()
        | (F.col("_order_id_clean") == ""),
        F.lit("MISSING_ORDER_ID")
    ).when(
        F.col("_pharmacy_id_clean").isNull()
        | (F.col("_pharmacy_id_clean") == ""),
        F.lit("MISSING_PHARMACY_ID")
    ).when(
        F.col("_known_pharmacy_id").isNull(),
        F.lit("UNKNOWN_PHARMACY_ID")
    ).when(
        F.col("order_time").isNull(),
        F.lit("INVALID_ORDER_TIME")
    ).when(
        F.col("promised_delivery_time").isNull(),
        F.lit("INVALID_PROMISED_DELIVERY_TIME")
    )
)

# Keep the original schema columns in both outputs.
original_columns = orders_bronze.columns

orders_silver_accepted = (
    orders_checked
    .filter(F.col("validation_reason").isNull())
    .select(*original_columns)
)

orders_silver_rejected = (
    orders_checked
    .filter(F.col("validation_reason").isNotNull())
    .select(
        *original_columns,
        "validation_reason"
    )
)

print("=== ORDER VALIDATION RESULTS ===")
print("Bronze orders:", orders_bronze.count())
print("Accepted orders:", orders_silver_accepted.count())
print("Rejected orders:", orders_silver_rejected.count())

print("\nRejection reasons:")
display(
    orders_silver_rejected
    .groupBy("validation_reason")
    .count()
    .orderBy("validation_reason")
)


=== ORDER VALIDATION RESULTS ===
Bronze orders: 2080
Accepted orders: 2040
Rejected orders: 40

Rejection reasons:


validation_reason,count
MISSING_ORDER_ID,20
UNKNOWN_PHARMACY_ID,20


In [0]:

# PHASE 4 - STEP 6: Validate and standardize order lines


# Use order IDs from accepted Silver orders.
# This excludes lines whose order was rejected.
valid_order_ids = (
    orders_silver_accepted
    .select(F.trim(F.col("order_id")).alias("_valid_order_id"))
    .distinct()
)

# Standardize fields and convert ordered quantity to BIGINT.
lines_prepared = (
    order_lines_bronze
    .withColumn("_order_id_clean", F.trim(F.col("order_id")))
    .withColumn("_line_id_clean", F.trim(F.col("line_id")))
    .withColumn(
        "ordered_quantity",
        F.expr("try_cast(ordered_quantity AS BIGINT)")
    )
    .join(
        valid_order_ids,
        F.col("_order_id_clean") == F.col("_valid_order_id"),
        "left"
    )
)

# Assign one primary rejection reason per record.
lines_checked = lines_prepared.withColumn(
    "validation_reason",
    F.when(
        F.col("_order_id_clean").isNull()
        | (F.col("_order_id_clean") == ""),
        F.lit("MISSING_ORDER_ID")
    ).when(
        F.col("_line_id_clean").isNull()
        | (F.col("_line_id_clean") == ""),
        F.lit("MISSING_LINE_ID")
    ).when(
        F.col("ordered_quantity").isNull()
        | (F.col("ordered_quantity") <= 0),
        F.lit("INVALID_ORDERED_QUANTITY")
    ).when(
        F.col("_valid_order_id").isNull(),
        F.lit("ORDER_NOT_ACCEPTED")
    )
)

# Keep original columns and their order.
original_line_columns = order_lines_bronze.columns

order_lines_silver_accepted = (
    lines_checked
    .filter(F.col("validation_reason").isNull())
    .select(*original_line_columns)
)

order_lines_silver_rejected = (
    lines_checked
    .filter(F.col("validation_reason").isNotNull())
    .select(
        *original_line_columns,
        "validation_reason"
    )
)

print("=== ORDER LINE VALIDATION RESULTS ===")
print("Bronze order lines:", order_lines_bronze.count())
print("Accepted order lines:", order_lines_silver_accepted.count())
print("Rejected order lines:", order_lines_silver_rejected.count())

print("\nRejection reasons:")
display(
    order_lines_silver_rejected
    .groupBy("validation_reason")
    .count()
    .orderBy("validation_reason")
)


=== ORDER LINE VALIDATION RESULTS ===
Bronze order lines: 4865
Accepted order lines: 4773
Rejected order lines: 92

Rejection reasons:


validation_reason,count
INVALID_ORDERED_QUANTITY,20
MISSING_LINE_ID,20
MISSING_ORDER_ID,20
ORDER_NOT_ACCEPTED,32


In [0]:


# Build the list of valid order-line combinations
valid_order_line_keys = (
    order_lines_silver_accepted
    .select(
        F.trim(F.col("order_id")).alias("_valid_order_id"),
        F.trim(F.col("line_id")).alias("_valid_line_id")
    )
    .distinct()
)

# Standardize fields for validation
shipment_lines_prepared = (
    shipment_lines_bronze
    .withColumn("_shipment_id_clean", F.trim(F.col("shipment_id")))
    .withColumn("_order_id_clean", F.trim(F.col("order_id")))
    .withColumn("_line_id_clean", F.trim(F.col("line_id")))
    .withColumn(
        "shipped_quantity",
        F.expr("try_cast(shipped_quantity AS BIGINT)")
    )
    .join(
        valid_order_line_keys,
        (
            (F.col("_order_id_clean") == F.col("_valid_order_id")) &
            (F.col("_line_id_clean") == F.col("_valid_line_id"))
        ),
        "left"
    )
)

# Assign one primary rejection reason to each invalid row
shipment_lines_checked = shipment_lines_prepared.withColumn(
    "validation_reason",
    F.when(
        F.col("_shipment_id_clean").isNull() |
        (F.col("_shipment_id_clean") == ""),
        F.lit("MISSING_SHIPMENT_ID")
    )
    .when(
        F.col("_order_id_clean").isNull() |
        (F.col("_order_id_clean") == ""),
        F.lit("MISSING_ORDER_ID")
    )
    .when(
        F.col("_line_id_clean").isNull() |
        (F.col("_line_id_clean") == ""),
        F.lit("MISSING_LINE_ID")
    )
    .when(
        F.col("shipped_quantity").isNull() |
        (F.col("shipped_quantity") <= 0),
        F.lit("INVALID_SHIPPED_QUANTITY")
    )
    .when(
        F.col("_valid_order_id").isNull(),
        F.lit("ORDER_LINE_NOT_ACCEPTED")
    )
)

# Separate accepted and rejected records
original_shipment_columns = shipment_lines_bronze.columns

shipment_lines_silver_accepted = (
    shipment_lines_checked
    .filter(F.col("validation_reason").isNull())
    .select(*original_shipment_columns)
)

shipment_lines_silver_rejected = (
    shipment_lines_checked
    .filter(F.col("validation_reason").isNotNull())
    .select(*original_shipment_columns, "validation_reason")
)

# Display results
print("=== SHIPMENT LINE VALIDATION RESULTS ===")
print("Bronze shipment lines:", shipment_lines_bronze.count())
print("Accepted shipment lines:", shipment_lines_silver_accepted.count())
print("Rejected shipment lines:", shipment_lines_silver_rejected.count())

print("\nRejection reasons:")
display(
    shipment_lines_silver_rejected
    .groupBy("validation_reason")
    .count()
    .orderBy("validation_reason")
)


=== SHIPMENT LINE VALIDATION RESULTS ===
Bronze shipment lines: 5958
Accepted shipment lines: 5958
Rejected shipment lines: 0

Rejection reasons:


validation_reason,count


In [0]:

# Delivery-event source was loaded in Step 2.
print("Bronze delivery event rows:", delivery_events_bronze.count())


Bronze delivery event rows: 6198


In [0]:

# 1. Prepare fields for validation
delivery_events_prepared = (
    delivery_events_bronze
    .withColumn("_event_id_clean", F.trim(F.col("event_id")))
    .withColumn("_shipment_id_clean", F.trim(F.col("shipment_id")))
    .withColumn("_order_id_clean", F.trim(F.col("order_id")))
    .withColumn("_line_id_clean", F.trim(F.col("line_id")))
    .withColumn(
        "delivered_quantity",
        F.expr("try_cast(delivered_quantity AS BIGINT)")
    )
    .withColumn(
        "delivery_time",
        F.expr("try_cast(delivery_time AS TIMESTAMP)")
    )
)

# 2. Assign a primary validation reason
delivery_events_checked = delivery_events_prepared.withColumn(
    "validation_reason",
    F.when(
        F.col("_event_id_clean").isNull() |
        (F.col("_event_id_clean") == ""),
        F.lit("MISSING_EVENT_ID")
    )
    .when(
        F.col("_shipment_id_clean").isNull() |
        (F.col("_shipment_id_clean") == ""),
        F.lit("MISSING_SHIPMENT_ID")
    )
    .when(
        F.col("_order_id_clean").isNull() |
        (F.col("_order_id_clean") == ""),
        F.lit("MISSING_ORDER_ID")
    )
    .when(
        F.col("_line_id_clean").isNull() |
        (F.col("_line_id_clean") == ""),
        F.lit("MISSING_LINE_ID")
    )
    .when(
        F.col("delivered_quantity").isNull() |
        (F.col("delivered_quantity") < 0),
        F.lit("INVALID_DELIVERED_QUANTITY")
    )
    .when(
        F.col("delivery_time").isNull(),
        F.lit("INVALID_DELIVERY_TIME")
    )
)

# 3. Separate records with valid fields from invalid records
delivery_events_field_valid = delivery_events_checked.filter(
    F.col("validation_reason").isNull()
)

delivery_events_field_rejected = delivery_events_checked.filter(
    F.col("validation_reason").isNotNull()
)

# 4. Deduplicate valid events by event_id.
# Keep the latest ingestion record; use source file as a tie-breaker.
dedup_window = Window.partitionBy("_event_id_clean").orderBy(
    F.col("_ingestion_timestamp").desc_nulls_last(),
    F.col("_source_file").desc_nulls_last()
)

delivery_events_ranked = delivery_events_field_valid.withColumn(
    "_event_rank",
    F.row_number().over(dedup_window)
)

delivery_events_deduplicated = (
    delivery_events_ranked
    .filter(F.col("_event_rank") == 1)
    .drop("_event_rank")
)

# 5. Keep duplicate copies separately for auditing
delivery_events_duplicate_rejected = (
    delivery_events_ranked
    .filter(F.col("_event_rank") > 1)
    .withColumn("validation_reason", F.lit("DUPLICATE_EVENT_ID"))
    .drop("_event_rank")
)

# 6. Show results
print("=== DELIVERY EVENT FIELD VALIDATION ===")
print("Bronze rows:", delivery_events_bronze.count())
print("Field-valid rows:", delivery_events_field_valid.count())
print("Field-invalid rows:", delivery_events_field_rejected.count())
print("Deduplicated valid rows:", delivery_events_deduplicated.count())
print("Duplicate copies:", delivery_events_duplicate_rejected.count())

print("\nField rejection reasons:")
display(
    delivery_events_field_rejected
    .groupBy("validation_reason")
    .count()
    .orderBy("validation_reason")
)


=== DELIVERY EVENT FIELD VALIDATION ===
Bronze rows: 6198
Field-valid rows: 6078
Field-invalid rows: 120
Deduplicated valid rows: 6078
Duplicate copies: 0

Field rejection reasons:


validation_reason,count
INVALID_DELIVERED_QUANTITY,60
MISSING_LINE_ID,30
MISSING_ORDER_ID,30


In [0]:

# 1. Get valid shipment-line keys
valid_shipment_line_keys = (
    shipment_lines_silver_accepted
    .select(
        F.trim(F.col("shipment_id")).alias("_valid_shipment_id"),
        F.trim(F.col("order_id")).alias("_valid_order_id"),
        F.trim(F.col("line_id")).alias("_valid_line_id")
    )
    .distinct()
)

# 2. Match field-valid, deduplicated events to shipment lines
delivery_events_matched = (
    delivery_events_deduplicated
    .join(
        valid_shipment_line_keys,
        (
            (F.col("_shipment_id_clean") == F.col("_valid_shipment_id")) &
            (F.col("_order_id_clean") == F.col("_valid_order_id")) &
            (F.col("_line_id_clean") == F.col("_valid_line_id"))
        ),
        "left"
    )
)

# 3. Separate accepted events from events to quarantine
delivery_events_silver_accepted = (
    delivery_events_matched
    .filter(F.col("_valid_shipment_id").isNotNull())
    .select(*delivery_events_bronze.columns)
)

delivery_events_quarantine = (
    delivery_events_matched
    .filter(F.col("_valid_shipment_id").isNull())
    .select(
        *delivery_events_bronze.columns,
        F.lit("UNKNOWN_SHIPMENT_LINE").alias("validation_reason")
    )
)

# 4. Show results
print("=== SHIPMENT-LINE MATCH RESULTS ===")
print("Deduplicated field-valid events:", delivery_events_deduplicated.count())
print("Accepted delivery events:", delivery_events_silver_accepted.count())
print("Events quarantined for unknown shipment line:", delivery_events_quarantine.count())

print("\nQuarantine reason summary:")
display(
    delivery_events_quarantine
    .groupBy("validation_reason")
    .count()
)


=== SHIPMENT-LINE MATCH RESULTS ===
Deduplicated field-valid events: 6078
Accepted delivery events: 6018
Events quarantined for unknown shipment line: 60

Quarantine reason summary:


validation_reason,count
UNKNOWN_SHIPMENT_LINE,60


In [0]:

# 1. Get accepted order quantities and order creation times
valid_order_lines_for_checks = (
    order_lines_silver_accepted
    .select(
        F.trim(F.col("order_id")).alias("_check_order_id"),
        F.trim(F.col("line_id")).alias("_check_line_id"),
        F.col("ordered_quantity").cast("long").alias("ordered_quantity")
    )
    .dropDuplicates(["_check_order_id", "_check_line_id"])
)

valid_orders_for_checks = (
    orders_silver_accepted
    .select(
        F.trim(F.col("order_id")).alias("_check_order_id"),
        F.col("order_time").cast("timestamp").alias("order_time")
    )
    .dropDuplicates(["_check_order_id"])
)

# 2. Sum accepted delivered quantities per order line
delivered_totals = (
    delivery_events_silver_accepted
    .groupBy(
        F.trim(F.col("order_id")).alias("_check_order_id"),
        F.trim(F.col("line_id")).alias("_check_line_id")
    )
    .agg(
        F.sum(F.col("delivered_quantity").cast("long")).alias("total_delivered_quantity")
    )
)

# 3. Join delivery events with order and line information
delivery_events_with_exceptions = (
    delivery_events_silver_accepted
    .withColumn("_check_order_id", F.trim(F.col("order_id")))
    .withColumn("_check_line_id", F.trim(F.col("line_id")))
    .withColumn("delivery_time", F.col("delivery_time").cast("timestamp"))
    .join(
        valid_order_lines_for_checks,
        ["_check_order_id", "_check_line_id"],
        "left"
    )
    .join(
        valid_orders_for_checks,
        "_check_order_id",
        "left"
    )
    .join(
        delivered_totals,
        ["_check_order_id", "_check_line_id"],
        "left"
    )
    .withColumn(
        "is_overdelivery",
        F.col("total_delivered_quantity") > F.col("ordered_quantity")
    )
    .withColumn(
        "is_delivery_before_order",
        F.col("delivery_time") < F.col("order_time")
    )
)

# 4. Summarize exception counts
print("=== DELIVERY EXCEPTION SUMMARY ===")
print(
    "Accepted delivery events:",
    delivery_events_with_exceptions.count()
)

print(
    "Events on lines with overdelivery:",
    delivery_events_with_exceptions
    .filter(F.col("is_overdelivery") == True)
    .count()
)

print(
    "Events delivered before order creation:",
    delivery_events_with_exceptions
    .filter(F.col("is_delivery_before_order") == True)
    .count()
)

print(
    "Events with either exception:",
    delivery_events_with_exceptions
    .filter(
        (F.col("is_overdelivery") == True) |
        (F.col("is_delivery_before_order") == True)
    )
    .count()
)

display(
    delivery_events_with_exceptions.select(
        "event_id",
        "order_id",
        "line_id",
        "delivered_quantity",
        "delivery_time",
        "ordered_quantity",
        "total_delivered_quantity",
        "is_overdelivery",
        "is_delivery_before_order"
    ).limit(20)
)


=== DELIVERY EXCEPTION SUMMARY ===
Accepted delivery events: 6018
Events on lines with overdelivery: 195
Events delivered before order creation: 60
Events with either exception: 195


event_id,order_id,line_id,delivered_quantity,delivery_time,ordered_quantity,total_delivered_quantity,is_overdelivery,is_delivery_before_order
EVT-0005404,ORD-000685,L03,4,2026-08-20T19:44:03.000Z,10,4,false,false
EVT-0001795,ORD-000694,L02,150,2026-08-19T06:00:20.000Z,150,150,false,false
EVT-0002532,ORD-000980,L01,12,2026-08-27T03:06:37.000Z,30,30,false,false
EVT-0005498,ORD-000980,L01,18,2026-08-28T21:20:48.000Z,30,30,false,false
EVT-0001503,ORD-000483,L01,10,2026-08-16T14:37:19.000Z,10,10,false,false
EVT-0000229,ORD-000142,L03,50,2026-08-04T23:50:28.000Z,50,50,false,false
EVT-0003449,ORD-001334,L03,76,2026-09-05T04:59:06.000Z,100,76,false,false
EVT-0001837,ORD-000676,L02,38,2026-08-19T15:40:16.000Z,60,43,false,false
EVT-0003919,ORD-001494,L01,14,2026-09-09T14:51:43.000Z,30,30,false,false
EVT-0004874,ORD-001879,L01,9,2026-09-17T23:55:08.000Z,20,20,false,false


In [0]:

# Standardized accepted orders
silver_orders_final = (
    orders_silver_accepted
    .select(
        F.trim(F.col("order_id")).alias("order_id"),
        F.trim(F.col("pharmacy_id")).alias("pharmacy_id"),
        F.col("order_time").cast("timestamp").alias("order_time"),
        F.col("promised_delivery_time").cast("timestamp").alias("promised_delivery_time"),
        F.col("order_status").alias("order_status"),
        "_source_file",
        "_ingestion_timestamp",
        "_batch_id"
    )
)

# Standardized accepted order lines
silver_order_lines_final = (
    order_lines_silver_accepted
    .select(
        F.trim(F.col("order_id")).alias("order_id"),
        F.trim(F.col("line_id")).alias("line_id"),
        F.col("product_id").alias("product_id"),
        F.col("ordered_quantity").cast("long").alias("ordered_quantity"),
        "_source_file",
        "_ingestion_timestamp",
        "_batch_id"
    )
)

# Standardized accepted shipment lines
silver_shipment_lines_final = (
    shipment_lines_silver_accepted
    .select(
        F.trim(F.col("shipment_id")).alias("shipment_id"),
        F.trim(F.col("order_id")).alias("order_id"),
        F.trim(F.col("line_id")).alias("line_id"),
        F.col("shipped_quantity").cast("long").alias("shipped_quantity"),
        "_source_file",
        "_ingestion_timestamp",
        "_batch_id"
    )
)

# Accepted delivery events with business exception flags
silver_delivery_events_final = (
    delivery_events_with_exceptions
    .select(
        F.trim(F.col("event_id")).alias("event_id"),
        F.trim(F.col("shipment_id")).alias("shipment_id"),
        F.trim(F.col("order_id")).alias("order_id"),
        F.trim(F.col("line_id")).alias("line_id"),
        F.col("delivered_quantity").cast("long").alias("delivered_quantity"),
        F.col("delivery_time").cast("timestamp").alias("delivery_time"),
        F.col("is_overdelivery").alias("is_overdelivery"),
        F.col("is_delivery_before_order").alias("is_delivery_before_order"),
        F.col("total_delivered_quantity").cast("long").alias("total_delivered_quantity"),
        "_source_file",
        "_ingestion_timestamp",
        "_batch_id"
    )
)

print("=== FINAL SILVER DATAFRAME COUNTS ===")
print("Silver orders:", silver_orders_final.count())
print("Silver order lines:", silver_order_lines_final.count())
print("Silver shipment lines:", silver_shipment_lines_final.count())
print("Silver delivery events:", silver_delivery_events_final.count())

print("\nSilver delivery event schema:")
silver_delivery_events_final.printSchema()


=== FINAL SILVER DATAFRAME COUNTS ===
Silver orders: 2040
Silver order lines: 4773
Silver shipment lines: 5958
Silver delivery events: 6018

Silver delivery event schema:
root
 |-- event_id: string (nullable = true)
 |-- shipment_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- line_id: string (nullable = true)
 |-- delivered_quantity: long (nullable = true)
 |-- delivery_time: timestamp (nullable = true)
 |-- is_overdelivery: boolean (nullable = true)
 |-- is_delivery_before_order: boolean (nullable = true)
 |-- total_delivered_quantity: long (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _batch_id: string (nullable = true)



In [0]:


pharmacies_checked = pharmacies_bronze.withColumn(
    "_pharmacy_id_clean",
    F.trim(F.col("pharmacy_id"))
).withColumn(
    "validation_reason",
    F.when(
        F.col("_pharmacy_id_clean").isNull() |
        (F.col("_pharmacy_id_clean") == ""),
        F.lit("MISSING_PHARMACY_ID")
    )
)

pharmacy_original_columns = pharmacies_bronze.columns

pharmacies_silver_final = (
    pharmacies_checked
    .filter(F.col("validation_reason").isNull())
    .select(
        F.col("_pharmacy_id_clean").alias("pharmacy_id"),
        "region",
        "contact_name",
        "contact_email",
        "_source_file",
        "_ingestion_timestamp",
        "_batch_id"
    )
)

pharmacies_silver_rejected = (
    pharmacies_checked
    .filter(F.col("validation_reason").isNotNull())
    .select(*pharmacy_original_columns, "validation_reason")
)

print("=== PHARMACY VALIDATION RESULTS ===")
print("Bronze pharmacies:", pharmacies_bronze.count())
print("Accepted pharmacies:", pharmacies_silver_final.count())
print("Rejected pharmacies:", pharmacies_silver_rejected.count())


=== PHARMACY VALIDATION RESULTS ===
Bronze pharmacies: 100
Accepted pharmacies: 100
Rejected pharmacies: 0


In [0]:

# 1. Rejected orders
silver_rejected_orders = orders_silver_rejected

# 2. Rejected order lines
silver_rejected_order_lines = order_lines_silver_rejected

# 3. Rejected delivery events with invalid fields
silver_rejected_delivery_events = (
    delivery_events_field_rejected
    .select(
        *delivery_events_bronze.columns,
        "validation_reason"
    )
)

# 4. Duplicate event copies, if any
silver_duplicate_delivery_events = (
    delivery_events_duplicate_rejected
    .select(
        *delivery_events_bronze.columns,
        "validation_reason"
    )
)

# 5. Delivery events with unknown shipment lines
silver_quarantined_delivery_events = delivery_events_quarantine

# 6. Review audit counts
print("=== AUDIT AND QUARANTINE COUNTS ===")
print("Rejected orders:", silver_rejected_orders.count())
print("Rejected order lines:", silver_rejected_order_lines.count())
print("Rejected delivery events:", silver_rejected_delivery_events.count())
print("Duplicate delivery-event copies:", silver_duplicate_delivery_events.count())
print("Quarantined delivery events:", silver_quarantined_delivery_events.count())

print("\nRejected order reasons:")
display(
    silver_rejected_orders
    .groupBy("validation_reason")
    .count()
    .orderBy("validation_reason")
)

print("\nRejected order-line reasons:")
display(
    silver_rejected_order_lines
    .groupBy("validation_reason")
    .count()
    .orderBy("validation_reason")
)

print("\nRejected delivery-event reasons:")
display(
    silver_rejected_delivery_events
    .groupBy("validation_reason")
    .count()
    .orderBy("validation_reason")
)


=== AUDIT AND QUARANTINE COUNTS ===
Rejected orders: 40
Rejected order lines: 92
Rejected delivery events: 120
Duplicate delivery-event copies: 0
Quarantined delivery events: 60

Rejected order reasons:


validation_reason,count
MISSING_ORDER_ID,20
UNKNOWN_PHARMACY_ID,20



Rejected order-line reasons:


validation_reason,count
INVALID_ORDERED_QUANTITY,20
MISSING_LINE_ID,20
MISSING_ORDER_ID,20
ORDER_NOT_ACCEPTED,32



Rejected delivery-event reasons:


validation_reason,count
INVALID_DELIVERED_QUANTITY,60
MISSING_LINE_ID,30
MISSING_ORDER_ID,30


In [0]:
# 12. Validate output counts and required columns before writing

tables_to_write = {
    "silver_pharmacies": pharmacies_silver_final,
    "silver_orders": silver_orders_final,
    "silver_order_lines": silver_order_lines_final,
    "silver_shipment_lines": silver_shipment_lines_final,
    "silver_delivery_events": silver_delivery_events_final,
    "silver_rejected_orders": silver_rejected_orders,
    "silver_rejected_order_lines": silver_rejected_order_lines,
    "silver_rejected_delivery_events": silver_rejected_delivery_events,
    "silver_duplicate_delivery_events": silver_duplicate_delivery_events,
    "silver_quarantined_delivery_events": silver_quarantined_delivery_events,
}

print("=== PRE-WRITE COUNTS ===")
for name, df in tables_to_write.items():
    print(f"{name}: {df.count()}")

# Safety checks: accepted delivery events must not contain duplicate event IDs.
duplicate_accepted_ids = (
    silver_delivery_events_final
    .groupBy("event_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)
if duplicate_accepted_ids:
    raise ValueError(
        f"Accepted Silver delivery events contain {duplicate_accepted_ids} duplicated event IDs."
    )

# Do not write if required output DataFrames are unexpectedly absent.
required_outputs = {
    "silver_pharmacies", "silver_orders", "silver_order_lines",
    "silver_shipment_lines", "silver_delivery_events"
}
if not required_outputs.issubset(set(tables_to_write.keys())):
    raise ValueError("One or more required Silver outputs are missing.")

print("Pre-write checks passed.")


=== PRE-WRITE COUNTS ===
silver_pharmacies: 100
silver_orders: 2040
silver_order_lines: 4773
silver_shipment_lines: 5958
silver_delivery_events: 6018
silver_rejected_orders: 40
silver_rejected_order_lines: 92
silver_rejected_delivery_events: 120
silver_duplicate_delivery_events: 0
silver_quarantined_delivery_events: 60
Pre-write checks passed.


In [0]:
# 13. Write accepted Silver and audit/quarantine tables

# This initial production job is a full refresh (input_batch='all').
# Bronze tables are never written to by this notebook.

for table_name, dataframe in tables_to_write.items():
    full_table_name = f"{TARGET}.{table_name}"
    (
        dataframe.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(full_table_name)
    )
    print(f"Saved: {full_table_name}")

print("All Silver and audit/quarantine tables saved.")


Saved: dbwteam3.order_fulfillment.silver_pharmacies
Saved: dbwteam3.order_fulfillment.silver_orders
Saved: dbwteam3.order_fulfillment.silver_order_lines
Saved: dbwteam3.order_fulfillment.silver_shipment_lines
Saved: dbwteam3.order_fulfillment.silver_delivery_events
Saved: dbwteam3.order_fulfillment.silver_rejected_orders
Saved: dbwteam3.order_fulfillment.silver_rejected_order_lines
Saved: dbwteam3.order_fulfillment.silver_rejected_delivery_events
Saved: dbwteam3.order_fulfillment.silver_duplicate_delivery_events
Saved: dbwteam3.order_fulfillment.silver_quarantined_delivery_events
All Silver and audit/quarantine tables saved.


In [0]:
# 14. Verify saved Silver tables against this run's actual counts

verification_results = []
for table_name, dataframe in tables_to_write.items():
    expected_count = dataframe.count()
    actual_count = spark.table(f"{TARGET}.{table_name}").count()
    status = "PASS" if actual_count == expected_count else "CHECK"
    verification_results.append(
        (table_name, expected_count, actual_count, status)
    )

verification_df = spark.createDataFrame(
    verification_results,
    ["table_name", "expected_rows", "actual_rows", "status"]
)
display(verification_df.orderBy("table_name"))

failed_count = sum(1 for row in verification_results if row[3] != "PASS")
print("Silver tables checked:", len(verification_results))
print("Tables requiring investigation:", failed_count)

if failed_count:
    raise RuntimeError("One or more saved Silver table counts do not match this run.")


table_name,expected_rows,actual_rows,status
silver_delivery_events,6018,6018,PASS
silver_duplicate_delivery_events,0,0,PASS
silver_order_lines,4773,4773,PASS
silver_orders,2040,2040,PASS
silver_pharmacies,100,100,PASS
silver_quarantined_delivery_events,60,60,PASS
silver_rejected_delivery_events,120,120,PASS
silver_rejected_order_lines,92,92,PASS
silver_rejected_orders,40,40,PASS
silver_shipment_lines,5958,5958,PASS


Silver tables checked: 10
Tables requiring investigation: 0


In [0]:
# 15. Verify Bronze source row counts were not changed by this notebook

bronze_results = []
for table_name, dataframe in [
    ("bronze_pharmacies", pharmacies_bronze),
    ("bronze_orders", orders_bronze),
    ("bronze_order_lines", order_lines_bronze),
    ("bronze_shipment_lines", shipment_lines_bronze),
    ("bronze_delivery_events", delivery_events_bronze),
]:
    expected_count = dataframe.count()
    actual_count = spark.table(f"{TARGET}.{table_name}").count()
    status = "PASS" if actual_count == expected_count else "CHECK"
    bronze_results.append((table_name, expected_count, actual_count, status))

bronze_verification_df = spark.createDataFrame(
    bronze_results,
    ["table_name", "rows_before", "rows_after", "status"]
)
display(bronze_verification_df.orderBy("table_name"))

bronze_failed = sum(1 for row in bronze_results if row[3] != "PASS")
print("Bronze tables checked:", len(bronze_results))
print("Bronze tables requiring investigation:", bronze_failed)

if bronze_failed:
    raise RuntimeError("A Bronze source row count changed unexpectedly.")

print("PHASE 4 SILVER JOB COMPLETED SUCCESSFULLY.")


table_name,rows_before,rows_after,status
bronze_delivery_events,6198,6198,PASS
bronze_order_lines,4865,4865,PASS
bronze_orders,2080,2080,PASS
bronze_pharmacies,100,100,PASS
bronze_shipment_lines,5958,5958,PASS


Bronze tables checked: 5
Bronze tables requiring investigation: 0
PHASE 4 SILVER JOB COMPLETED SUCCESSFULLY.


In [0]:

# NEW RELIC — SILVER MONITORING INITIALIZATION
# Add this cell after Section 1: Job parameters and shared configuration.

import time
from datetime import datetime, timezone

silver_pipeline_name = "order_fulfilment"
silver_stage = "silver"

silver_start_time = time.time()
silver_status = "success"
silver_error = None

# Input counts from Bronze sources
silver_input_rows = 0

# Records rejected by validation
silver_rejected_rows = 0

# Output and quality metrics
silver_output_rows = 0
silver_validation_failures = 0
silver_quarantined_rows = 0

# Freshness is calculated separately after outputs are saved.
silver_freshness_seconds = None

print("Silver monitoring initialized.")


Silver monitoring initialized.


In [0]:

# NEW RELIC — CAPTURE SILVER INPUT COUNTS

silver_bronze_counts = {
    "bronze_pharmacies": pharmacies_bronze.count(),
    "bronze_orders": orders_bronze.count(),
    "bronze_order_lines": order_lines_bronze.count(),
    "bronze_shipment_lines": shipment_lines_bronze.count(),
    "bronze_delivery_events": delivery_events_bronze.count()
}

silver_input_rows = sum(silver_bronze_counts.values())

print("Silver input rows across all Bronze sources:", silver_input_rows)
print("Per-table input counts:", silver_bronze_counts)


Silver input rows across all Bronze sources: 19201
Per-table input counts: {'bronze_pharmacies': 100, 'bronze_orders': 2080, 'bronze_order_lines': 4865, 'bronze_shipment_lines': 5958, 'bronze_delivery_events': 6198}
